# assignment 6 - deep feelings
## sentiment analysis of tweers using nlp
### cs 562 - spring 2026 (undergraduate)

In [1]:
# imports

import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

## 1. data loading

In [5]:
# load data
train_df = pd.read_csv('A06_train.csv', header=None, names=['label', 'text'])
test_df = pd.read_csv('A06_test.csv', header=None, names=['label','text'])

print(f"trains: {len(train_df)} rows")
print(f"test: {len(test_df)} rows")
print(train_df['label'].value_counts())

trains: 26499 rows
test: 6625 rows
label
 0    11510
 1     9621
-1     5368
Name: count, dtype: int64


## 2. text cleaning

remove all urls, @mentions, html, and non-alphabetic characters. text is lowercased to reduce noise in the data

In [6]:
# clean text
def clean_text(text):
    # remove urls
    text = re.sub(r'http\S+', '', text)
    # remove @mentions
    text = re.sub(r'@\w+', '', text)
    # remove html entities
    text = re.sub(r'&\w+', '', text)
    # keep only letters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    return text

train_df['clean'] = train_df['text'].apply(clean_text)
test_df['clean'] = test_df['text'].apply(clean_text)

## 3. baseline system

baseline uses bag-of-words features (CountVectorizer) with a LinearSVC classifier

In [7]:
# baseline model
vectorizer = CountVectorizer()
x_train = vectorizer.fit_transform(train_df['clean'])
x_test = vectorizer.transform(test_df['clean'])

y_train = train_df['label']
y_test = test_df['label']

svm = LinearSVC()
svm.fit(x_train, y_train)

y_pred = svm.predict(x_test)

print('=== baseline system ===')
print(f'accuraccy: {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'poositive']))

=== baseline system ===
accuraccy: 0.6222

              precision    recall  f1-score   support

    negative       0.57      0.52      0.54      1319
     neutral       0.61      0.65      0.63      2877
   poositive       0.66      0.65      0.66      2429

    accuracy                           0.62      6625
   macro avg       0.61      0.60      0.61      6625
weighted avg       0.62      0.62      0.62      6625



## 4. enchanced system

the baseline is enhanced with all three augmentations: stopword removal, lemmatization, and bigram features

In [10]:
# enchanced imports
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_enhanced(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'&\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    # tokenize, remove stopwords, lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

train_df['clean_enhanced'] = train_df['text'].apply(clean_text_enhanced)
test_df['clean_enhanced'] = test_df['text'].apply(clean_text_enhanced)

In [12]:
# enhanced model
vectorizer_enhanced = CountVectorizer(ngram_range=(1,2))
x_train_enh = vectorizer_enhanced.fit_transform(train_df['clean_enhanced'])
x_test_enh = vectorizer_enhanced.transform(test_df['clean_enhanced'])

svm_enhanced = LinearSVC()
svm_enhanced.fit(x_train_enh, y_train)

y_pred_enh = svm_enhanced.predict(x_test_enh)

print("=== enhanced system ===")
print(f"accuracy: {accuracy_score(y_test, y_pred_enh):.4f}")
print()
print(classification_report(y_test, y_pred_enh, target_names=['negative', 'neutral', 'positive']))

=== enhanced system ===
accuracy: 0.6478

              precision    recall  f1-score   support

    negative       0.63      0.48      0.54      1319
     neutral       0.62      0.71      0.66      2877
    positive       0.70      0.67      0.68      2429

    accuracy                           0.65      6625
   macro avg       0.65      0.62      0.63      6625
weighted avg       0.65      0.65      0.65      6625



### 4.1 individual enhancement analysis

test each enhancement individually to determine its contribution to performance

In [18]:
# stopwords only
def clean_stopwords(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'&\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    tokens = [t for t in text.split() if t not in stop_words]
    return ' '.join(tokens)

train_df['clean_sw'] = train_df['text'].apply(clean_stopwords)
test_df['clean_sw'] = test_df['text'].apply(clean_stopwords)

vec_sw = CountVectorizer()
svm_sw = LinearSVC()
svm_sw.fit(vec_sw.fit_transform(train_df['clean_sw']), y_train)
y_pred_sw = svm_sw.predict(vec_sw.transform(test_df['clean_sw']))
print(f"stopwords only accuracy: {accuracy_score(y_test, y_pred_sw):.4f}")

stopwords only accuracy: 0.6136


In [19]:
# lemmatization only
def clean_lemma(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    tokens = [lemmatizer.lemmatize(t) for t in text.split()]
    return ' '.join(tokens)

train_df['clean_lemma'] = train_df['text'].apply(clean_lemma)
test_df['clean_lemma'] = test_df['text'].apply(clean_lemma)

vec_lemma = CountVectorizer()
svm_lemma = LinearSVC()
svm_lemma.fit(vec_lemma.fit_transform(train_df['clean_lemma']), y_train)
y_pred_lemma = svm_lemma.predict(vec_lemma.transform(test_df['clean_lemma']))
print(f"lemmatization only accuracy: {accuracy_score(y_test, y_pred_lemma):.4f}")

lemmatization only accuracy: 0.6180


c:\Users\valar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [20]:
# bigrams only
vec_bigram = CountVectorizer(ngram_range=(1, 2))
svm_bigram = LinearSVC()
svm_bigram.fit(vec_bigram.fit_transform(train_df['clean']), y_train)
y_pred_bigram = svm_bigram.predict(vec_bigram.transform(test_df['clean']))
print(f"bigrams only accuracy: {accuracy_score(y_test, y_pred_bigram):.4f}")

bigrams only accuracy: 0.6380


## 5. embeddings system

baseline is enhanced by replacing bag-of-words features with spaCy word vectors. each tweet is represented as the average of its token vector

In [15]:
# load spaCy model
import spacy
import numpy as np

nlp = spacy.load('en_core_web_md')

def get_embedding(text):
    doc = nlp(text)
    return doc.vector

train_df['embedding'] = train_df['clean'].apply(get_embedding)
test_df['embedding'] = test_df['clean'].apply(get_embedding)

print('embeddings done')

embeddings done


In [17]:
# embeddings model
x_train_emb = np.vstack(train_df['embedding'].values)
x_test_emb = np.stack(test_df['embedding'].values)

svm_emb = LinearSVC()
svm_emb.fit(x_train_emb, y_train)

y_pred_emb = svm_emb.predict(x_test_emb)

print("=== embeddings system ===")
print(f"accuracy: {accuracy_score(y_test, y_pred_emb):.4f}")
print()
print(classification_report(y_test, y_pred_emb, target_names=['negative', 'neutral', 'positive']))

=== embeddings system ===
accuracy: 0.5583

              precision    recall  f1-score   support

    negative       0.57      0.35      0.43      1319
     neutral       0.55      0.64      0.59      2877
    positive       0.56      0.58      0.57      2429

    accuracy                           0.56      6625
   macro avg       0.56      0.52      0.53      6625
weighted avg       0.56      0.56      0.55      6625

